# 03 — Model Training
Train Random Forest and XGBoost models. Select the best via cross-validation.

In [ ]:
import sys, os
sys.path.append(os.path.dirname(os.getcwd()))
import pandas as pd
import config
from src.features import add_behavioral_features
from src.train import train_random_forest, train_xgboost, cross_validate, save_model
from src.evaluate import evaluate_model, print_report

## 1. Load Processed Data

In [ ]:
X_train = pd.read_csv(config.X_TRAIN_PROCESSED)
y_train = pd.read_csv(config.Y_TRAIN_PATH).squeeze()
print('X_train:', X_train.shape, '| Fraud rate:', round(y_train.mean()*100, 2), '%')

## 2. Feature Engineering

In [ ]:
X_train = add_behavioral_features(X_train)
print('Features after engineering:', X_train.shape[1])

## 3. Train Random Forest

In [ ]:
rf_model = train_random_forest(X_train, y_train)
rf_auc   = cross_validate(rf_model, X_train, y_train, cv=5)
print(f'Random Forest CV AUC: {rf_auc:.4f}')

## 4. Train XGBoost

In [ ]:
xgb_model = train_xgboost(X_train, y_train)
xgb_auc   = cross_validate(xgb_model, X_train, y_train, cv=5)
print(f'XGBoost CV AUC: {xgb_auc:.4f}')

## 5. Compare & Save Best Model

In [ ]:
best_model      = rf_model if rf_auc >= xgb_auc else xgb_model
best_name       = 'Random Forest' if rf_auc >= xgb_auc else 'XGBoost'
print(f'Best: {best_name} (AUC = {max(rf_auc, xgb_auc):.4f})')
save_model(best_model)
print('Model saved.')

## 6. Training Set Report

In [ ]:
y_pred = best_model.predict(X_train)
metrics = evaluate_model(y_train, y_pred, best_model.predict_proba(X_train)[:, 1])
print('\nMetrics:', metrics)
print()
print_report(y_train, y_pred)